In [ ]:
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
from scipy.signal import detrend as signal_detrend
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA


EVAP_EXCEL_PATH = Path("data") / "evaporation_data.xlsx"
SIMCA_WORKBOOK = Path("data") / "data.xlsx"
OUTPUT_DIR = Path("outputs") / "evaporation"

EVAP_TRAIN_SHEET = "train"
EVAP_TEST_SHEET = "test"
POSITIVE_SHEET = "Positive"

MAX_COMPONENTS = 3
MASS_GROUP_DECIMALS = 4

CLASSES = ["Lamp oil", "White spirit", "Diesel", "Gasoline", "Brandspiritus"]
CONFIDENCE_LEVELS = [0.9999]

METHOD_ORDER = [
    "raw",
    "baseline",
    "detrend",
    "normalisation",
    "snv",
    "msc",
    "sg1_p2_w3",
    "sg2_p2_w3",
    "snv+sg1",
    "msc+sg1",
]

PRETTY_NAMES = {
    "raw": "Raw",
    "baseline": "Baseline correction",
    "detrend": "Detrending",
    "normalisation": "Normalisation",
    "snv": "SNV",
    "msc": "MSC",
    "sg1_p2_w3": "SG1 (p2, w3)",
    "sg2_p2_w3": "SG2 (p2, w3)",
    "snv+sg1": "SNV + SG1",
    "msc+sg1": "MSC + SG1",
}

LEVEL1_SCENARIO1_TEST_ROOTS = {
    "T1", "T2", "T4", "T5", "T7", "T10", "T11", "T13", "T14",
    "T6", "T9", "T12", "T15", "Te3", "Te6", "W1", "W2", "W3",
    "L1", "L3", "L7",
}

DIESEL_ROOTS = {
    "SH3", "SH4", "SH7", "SH8", "SH11", "SH12", "SH15", "SH16",
    "SH19", "SH20", "T3", "T6", "T9", "T12", "T15", "Te3", "Te6",
    "Te9", "Te12", "Te15",
}

GAS95_ROOTS = {
    "SH1", "SH5", "SH9", "SH13", "SH17", "T1", "T4", "T7", "T10",
    "T13", "Te1", "Te4", "Te7", "Te10", "Te13",
}

GAS98_ROOTS = {
    "SH2", "SH6", "SH10", "SH14", "SH18", "T2", "T5", "T8", "T11",
    "T14", "Te2", "Te5", "Te8", "Te11", "Te14",
}

MASS_COL_CANDIDATES = [
    "mass (%)",
    "mass%",
    "mass",
    "remaining mass (%)",
    "remaining_mass_pct",
    "remaining mass",
    "mass of loss (%)",
    "mass loss (%)",
]


def root_code(sample_id: str) -> str:
    return str(sample_id).split("-", 1)[0]


def confidence_label(confidence: float) -> str:
    percent = confidence * 100.0
    if abs(percent - round(percent)) < 1e-10:
        return f"{int(round(percent))}%"
    return f"{percent:.2f}".rstrip("0").rstrip(".") + "%"


def format_mass_label(value: float) -> str:
    text = f"{float(value):.{MASS_GROUP_DECIMALS}f}".rstrip("0").rstrip(".")
    return text + "%"


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalised = {str(column).strip().lower(): column for column in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalised:
            return normalised[key]

    for column in df.columns:
        column_key = str(column).strip().lower()
        for candidate in candidates:
            if column_key.startswith(candidate.strip().lower()):
                return column
    return None


def get_spectral_columns_and_axis(
    df: pd.DataFrame,
    exclude_cols: set[str] | None = None,
) -> tuple[list[str], np.ndarray]:
    excluded = exclude_cols or set()
    numeric_columns = []
    wavelengths = []

    for column in df.columns:
        if column in excluded:
            continue
        try:
            wavelength = float(str(column).strip())
        except (TypeError, ValueError):
            continue
        numeric_columns.append(column)
        wavelengths.append(wavelength)

    if not numeric_columns:
        raise ValueError("No numeric spectral columns were found.")

    axis = np.asarray(wavelengths, dtype=float)
    order = np.argsort(axis)
    return [numeric_columns[i] for i in order], axis[order]


def align_spectral_dataframes(
    development: pd.DataFrame,
    development_wavelengths: np.ndarray,
    evaporation: pd.DataFrame,
    evaporation_wavelengths: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    development_map = {
        round(float(wavelength), 6): column
        for wavelength, column in zip(development_wavelengths, development.columns)
    }
    evaporation_map = {
        round(float(wavelength), 6): column
        for wavelength, column in zip(evaporation_wavelengths, evaporation.columns)
    }
    common = sorted(set(development_map).intersection(evaporation_map))
    if not common:
        raise ValueError("No common wavelength columns were found.")

    development = development.loc[:, [development_map[key] for key in common]].copy()
    evaporation = evaporation.loc[:, [evaporation_map[key] for key in common]].copy()
    column_names = [f"{value:.6f}".rstrip("0").rstrip(".") for value in common]
    development.columns = column_names
    evaporation.columns = column_names
    return development, evaporation, np.asarray(common, dtype=float)


def remaining_to_mass_loss_pct(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return values.copy()
    if np.nanmax(np.abs(finite)) <= 1.5:
        return (1.0 - values) * 100.0
    return 100.0 - values


def build_metadata(spectra: pd.DataFrame) -> pd.DataFrame:
    roots = spectra.index.to_series().astype(str).map(root_code)
    classes = []

    for root in roots:
        if root.startswith("L"):
            classes.append("Lamp oil")
        elif root.startswith("W"):
            classes.append("White spirit")
        elif root in DIESEL_ROOTS:
            classes.append("Diesel")
        elif root in GAS95_ROOTS or root in GAS98_ROOTS:
            classes.append("Gasoline")
        elif root.startswith("B"):
            classes.append("Brandspiritus")
        else:
            raise ValueError(f"Unknown root code in Positive sheet: {root}")

    return pd.DataFrame({"root": roots.to_numpy(), "simca_class": classes}, index=spectra.index)


def load_simca_development_set() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    positive = pd.read_excel(SIMCA_WORKBOOK, sheet_name=POSITIVE_SHEET)
    positive = positive.set_index(positive.columns[0])
    positive.index = positive.index.astype(str)
    positive.columns = positive.columns.astype(str)

    spectral_columns, wavelengths = get_spectral_columns_and_axis(positive)
    spectra = positive.loc[:, spectral_columns].copy()
    metadata = build_metadata(spectra)

    available_roots = set(metadata["root"].unique())
    missing_roots = sorted(LEVEL1_SCENARIO1_TEST_ROOTS - available_roots)
    if missing_roots:
        raise ValueError(f"Level 1 external-test roots missing from Positive sheet: {missing_roots}")

    development_mask = ~metadata["root"].isin(LEVEL1_SCENARIO1_TEST_ROOTS)
    development = spectra.loc[development_mask].copy()
    development_metadata = metadata.loc[development_mask].copy()
    if development.empty:
        raise ValueError("The Level 1 development set is empty.")

    return development, development_metadata, wavelengths


def load_evaporation_data() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    workbook = pd.ExcelFile(EVAP_EXCEL_PATH)
    expected_sheets = [EVAP_TRAIN_SHEET, EVAP_TEST_SHEET]
    missing_sheets = [sheet for sheet in expected_sheets if sheet not in workbook.sheet_names]
    if missing_sheets:
        raise ValueError(f"Missing evaporation sheet(s): {missing_sheets}")

    train = pd.read_excel(EVAP_EXCEL_PATH, sheet_name=EVAP_TRAIN_SHEET, index_col=0)
    test = pd.read_excel(EVAP_EXCEL_PATH, sheet_name=EVAP_TEST_SHEET, index_col=0)
    combined = pd.concat([train, test], axis=0)
    combined.index = combined.index.astype(str)
    combined.columns = combined.columns.astype(str)

    mass_column = find_column(combined, MASS_COL_CANDIDATES)
    if mass_column is None:
        mass_column = combined.columns[0]

    remaining_mass = pd.to_numeric(combined[mass_column], errors="coerce").to_numpy(dtype=float)
    if np.isnan(remaining_mass).any():
        raise ValueError("The remaining-mass column contains missing or non-numeric values.")

    mass_loss = np.round(
        remaining_to_mass_loss_pct(remaining_mass),
        MASS_GROUP_DECIMALS,
    )
    spectral_columns, wavelengths = get_spectral_columns_and_axis(
        combined,
        exclude_cols={mass_column},
    )
    spectra = combined.loc[:, spectral_columns].copy()
    if spectra.isna().any().any():
        spectra = spectra.apply(lambda column: column.fillna(column.mean()), axis=0)

    metadata = pd.DataFrame(
        {
            "Mass of loss (%)": mass_loss,
            "Mass label": [format_mass_label(value) for value in mass_loss],
        },
        index=combined.index,
    )
    return spectra, metadata, wavelengths


def preproc_raw(df: pd.DataFrame) -> pd.DataFrame:
    return df.copy()


def preproc_normalisation(df: pd.DataFrame) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    norms[norms == 0.0] = 1.0
    return pd.DataFrame(values / norms, index=df.index, columns=df.columns)


def preproc_snv(df: pd.DataFrame) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    means = values.mean(axis=1, keepdims=True)
    standard_deviations = values.std(axis=1, ddof=1, keepdims=True)
    standard_deviations[standard_deviations == 0.0] = 1.0
    corrected = (values - means) / standard_deviations
    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_msc(df: pd.DataFrame, reference: np.ndarray) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    reference = np.asarray(reference, dtype=float)
    design = np.column_stack([reference, np.ones_like(reference)])
    corrected = np.empty_like(values)

    for index, row in enumerate(values):
        slope, intercept = np.linalg.lstsq(design, row, rcond=None)[0]
        corrected[index] = (row - intercept) / slope if slope != 0 else row - intercept

    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_baseline_endpoints(df: pd.DataFrame, wavelengths: np.ndarray) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    denominator = wavelengths[-1] - wavelengths[0]
    if denominator == 0:
        denominator = 1.0
    corrected = np.empty_like(values)

    for index, row in enumerate(values):
        slope = (row[-1] - row[0]) / denominator
        baseline = row[0] + slope * (wavelengths - wavelengths[0])
        corrected[index] = row - baseline

    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_detrend(df: pd.DataFrame) -> pd.DataFrame:
    corrected = signal_detrend(df.to_numpy(dtype=float), axis=1, type="linear")
    return pd.DataFrame(corrected, index=df.index, columns=df.columns)


def preproc_sg(
    df: pd.DataFrame,
    polyorder: int,
    window_length: int,
    deriv: int,
) -> pd.DataFrame:
    values = df.to_numpy(dtype=float)
    if window_length % 2 == 0:
        window_length += 1
    if window_length > values.shape[1]:
        window_length = values.shape[1] if values.shape[1] % 2 == 1 else values.shape[1] - 1
    filtered = savgol_filter(
        values,
        window_length=window_length,
        polyorder=polyorder,
        deriv=deriv,
        axis=1,
        mode="interp",
    )
    return pd.DataFrame(filtered, index=df.index, columns=df.columns)


def get_preprocessing_pipelines(
    wavelengths: np.ndarray,
    msc_reference: np.ndarray,
) -> dict[str, Callable[[pd.DataFrame], pd.DataFrame]]:
    def sg1(df: pd.DataFrame) -> pd.DataFrame:
        return preproc_sg(df, polyorder=2, window_length=3, deriv=1)

    def sg2(df: pd.DataFrame) -> pd.DataFrame:
        return preproc_sg(df, polyorder=2, window_length=3, deriv=2)

    return {
        "raw": preproc_raw,
        "baseline": lambda df: preproc_baseline_endpoints(df, wavelengths),
        "detrend": preproc_detrend,
        "normalisation": preproc_normalisation,
        "snv": preproc_snv,
        "msc": lambda df: preproc_msc(df, msc_reference),
        "sg1_p2_w3": sg1,
        "sg2_p2_w3": sg2,
        "snv+sg1": lambda df: sg1(preproc_snv(df)),
        "msc+sg1": lambda df: sg1(preproc_msc(df, msc_reference)),
    }


class SimcaClassModel:
    def __init__(
        self,
        pca: PCA,
        mean: np.ndarray,
        eigenvalues: np.ndarray,
        training_t2: np.ndarray,
        training_q: np.ndarray,
    ) -> None:
        self.pca = pca
        self.mean = mean
        self.eigenvalues = eigenvalues
        self.training_t2 = training_t2
        self.training_q = training_q


def fit_simca_class(values: np.ndarray) -> SimcaClassModel:
    n_components = min(MAX_COMPONENTS, values.shape[0], values.shape[1])
    mean = values.mean(axis=0)
    centred = values - mean
    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(centred)
    reconstructed = pca.inverse_transform(scores)
    residuals = centred - reconstructed
    eigenvalues = pca.explained_variance_
    training_t2 = np.sum((scores ** 2) / eigenvalues, axis=1)
    training_q = np.sum(residuals ** 2, axis=1)
    return SimcaClassModel(pca, mean, eigenvalues, training_t2, training_q)


def fit_all_class_models(
    development: pd.DataFrame,
    metadata: pd.DataFrame,
) -> dict[str, SimcaClassModel]:
    models = {}
    for class_name in CLASSES:
        class_values = development.loc[
            metadata["simca_class"] == class_name
        ].to_numpy(dtype=float)
        if class_values.shape[0] == 0:
            raise ValueError(f"No development samples available for class: {class_name}")
        models[class_name] = fit_simca_class(class_values)
    return models


def score_simca_class(
    values: np.ndarray,
    model: SimcaClassModel,
) -> tuple[np.ndarray, np.ndarray]:
    centred = values - model.mean
    scores = model.pca.transform(centred)
    reconstructed = model.pca.inverse_transform(scores)
    residuals = centred - reconstructed
    t2 = np.sum((scores ** 2) / model.eigenvalues, axis=1)
    q = np.sum(residuals ** 2, axis=1)
    return t2, q


def classify_with_simca(
    spectra: pd.DataFrame,
    models: dict[str, SimcaClassModel],
    alpha: float,
) -> pd.DataFrame:
    values = spectra.to_numpy(dtype=float)
    membership = pd.DataFrame(False, index=spectra.index, columns=CLASSES)

    for class_name in CLASSES:
        model = models[class_name]
        t2, q = score_simca_class(values, model)
        t2_limit = np.quantile(model.training_t2, 1.0 - alpha)
        q_limit = np.quantile(model.training_q, 1.0 - alpha)
        membership[class_name] = (t2 <= t2_limit) & (q <= q_limit)

    return membership


def membership_to_accept_refuse(membership: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            f"{class_name} PCA": np.where(membership[class_name], "accept", "refuse")
            for class_name in CLASSES
        },
        index=membership.index,
    )


def build_probability_summary(per_spectrum: pd.DataFrame) -> pd.DataFrame:
    result_rows = []
    group_columns = [
        "Scenario",
        "Preprocessing method",
        "Confidence level",
        "Confidence level numeric",
        "Method order",
        "Mass of loss (%)",
        "Mass label",
    ]

    for keys, group in per_spectrum.groupby(group_columns, sort=True):
        row = dict(zip(group_columns, keys))
        row["n spectra"] = len(group)
        for class_name in CLASSES:
            column = f"{class_name} PCA"
            accept_probability = float((group[column] == "accept").mean())
            row[f"{column} accept probability"] = accept_probability
            row[f"{column} refuse probability"] = 1.0 - accept_probability
        result_rows.append(row)

    summary = pd.DataFrame(result_rows)
    summary = summary.sort_values(
        ["Confidence level numeric", "Method order", "Mass of loss (%)"],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return summary.drop(columns=["Confidence level numeric", "Method order"])


def run() -> None:
    development, development_metadata, development_wavelengths = load_simca_development_set()
    evaporation, evaporation_metadata, evaporation_wavelengths = load_evaporation_data()
    development, evaporation, wavelengths = align_spectral_dataframes(
        development,
        development_wavelengths,
        evaporation,
        evaporation_wavelengths,
    )

    msc_reference = development.to_numpy(dtype=float).mean(axis=0)
    pipelines = get_preprocessing_pipelines(wavelengths, msc_reference)
    all_results = []

    for method_order, method in enumerate(METHOD_ORDER):
        preprocessing = pipelines[method]
        prepared_development = preprocessing(development)
        prepared_evaporation = preprocessing(evaporation)
        models = fit_all_class_models(prepared_development, development_metadata)

        for confidence in CONFIDENCE_LEVELS:
            membership = classify_with_simca(
                prepared_evaporation,
                models,
                alpha=1.0 - confidence,
            )
            frame = pd.DataFrame(
                {
                    "Scenario": f"{PRETTY_NAMES[method]}, {confidence_label(confidence)}",
                    "Preprocessing method": PRETTY_NAMES[method],
                    "Confidence level": confidence_label(confidence),
                    "Confidence level numeric": confidence,
                    "Method order": method_order,
                    "Mass of loss (%)": evaporation_metadata["Mass of loss (%)"].to_numpy(),
                    "Mass label": evaporation_metadata["Mass label"].to_numpy(),
                }
            )
            frame = pd.concat(
                [frame, membership_to_accept_refuse(membership).reset_index(drop=True)],
                axis=1,
            )
            all_results.append(frame)

    probability_summary = build_probability_summary(
        pd.concat(all_results, ignore_index=True)
    )
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / "evaporation_SIMCA.xlsx"
    probability_summary.to_excel(
        output_path,
        sheet_name="mass_probabilities",
        index=False,
    )
    print(f"Saved: {output_path}")


if __name__ == "__main__":
    run()
